In [ ]:
!pip install transformers datasets accelerate evaluate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00


In [ ]:
!pip install transformers datasets evaluate accelerate -q

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from datasets import Dataset

In [ ]:
df = pd.read_csv("/content/sentiment_real_reviews_1000.csv")

df = df[["reviews", "sentiment_label"]].dropna()
df = df.rename(columns={"reviews": "text", "sentiment_label": "label"})

print(df.head())
print(df["label"].value_counts())

                                                text     label
0  Lol.... the only reason i bought this tablet w...  negative
1  Wish tablet was voice capable. Only problem wa...  negative
2  Alexa works great with music. Just wish you di...  positive
3  Led to more echo dots and significant home aut...   neutral
4  I purchased this after trying the Insignia 8" ...  positive
label
negative    334
positive    333
neutral     333
Name: count, dtype: int64


In [ ]:
label2id = {"negative": 0, "neutral": 1, "positive": 2}
id2label = {0: "negative", 1: "neutral", 2: "positive"}

df["label"] = df["label"].map(label2id)

In [ ]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [ ]:
train_dataset = train_dataset.rename_column("label", "labels")
test_dataset = test_dataset.rename_column("label", "labels")

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
import evaluate

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels, average="weighted")["f1"]
    }

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./bert_sentiment",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch", # Added this line to match eval_strategy
    learning_rate=2e-5,
    load_best_model_at_end=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.983237,0.689199,0.730000,0.707704
2,0.439730,0.409023,0.860000,0.854872
3,0.180940,0.298090,0.915000,0.913775
4,0.072709,0.312674,0.915000,0.913356


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=400, training_loss=0.41915376782417296, metrics={'train_runtime': 126.9139, 'train_samples_per_second': 25.214, 'train_steps_per_second': 3.152, 'total_flos': 210490734182400.0, 'train_loss': 0.41915376782417296, 'epoch': 4.0})

In [ ]:
results=trainer.evaluate()
print(results)

{'eval_loss': 0.2980356514453888, 'eval_accuracy': 0.915, 'eval_f1': 0.9137745015915564, 'eval_runtime': 1.6794, 'eval_samples_per_second': 119.09, 'eval_steps_per_second': 14.886, 'epoch': 4.0}


In [ ]:
predictions = trainer.predict(test_dataset)

y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_true, y_pred, target_names=["negative", "neutral", "positive"]))
print(confusion_matrix(y_true, y_pred))

              precision    recall  f1-score   support

    negative       0.89      0.99      0.94        67
     neutral       0.92      0.82      0.87        67
    positive       0.94      0.94      0.94        66

    accuracy                           0.92       200
   macro avg       0.92      0.92      0.91       200
weighted avg       0.92      0.92      0.91       200

[[66  1  0]
 [ 8 55  4]
 [ 0  4 62]]


In [ ]:
import torch

def predict_sentiment(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    # Move inputs to the same device as the model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)

    predicted_class = torch.argmax(outputs.logits, dim=1).item()
    return id2label[predicted_class]

sample_reviews = [
    "This product is excellent and works perfectly. I am very happy with it.",
    "The item is okay. Not bad, but nothing special.",
    "Very disappointing product. It stopped working after two days."
]

for review in sample_reviews:
    print("Review:", review)
    print("Predicted sentiment:", predict_sentiment(review))
    print("-" * 60)

Review: This product is excellent and works perfectly. I am very happy with it.
Predicted sentiment: positive
------------------------------------------------------------
Review: The item is okay. Not bad, but nothing special.
Predicted sentiment: negative
------------------------------------------------------------
Review: Very disappointing product. It stopped working after two days.
Predicted sentiment: negative
------------------------------------------------------------


In [ ]:
model.save_pretrained("/content/bert_sarcasm_model")
tokenizer.save_pretrained("/content/bert_sarcasm_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/bert_sarcasm_model/tokenizer_config.json',
 '/content/bert_sarcasm_model/tokenizer.json')

Testing Zero-shot model for sentiment analysis

In [ ]:
!pip install transformers -q

from transformers import pipeline
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
df = pd.read_csv("/content/sentiment_real_reviews_1000.csv")

df = df[["reviews", "sentiment_label"]].dropna()

print(df.head())

                                             reviews sentiment_label
0  Lol.... the only reason i bought this tablet w...        negative
1  Wish tablet was voice capable. Only problem wa...        negative
2  Alexa works great with music. Just wish you di...        positive
3  Led to more echo dots and significant home aut...         neutral
4  I purchased this after trying the Insignia 8" ...        positive


In [ ]:
labels = ["positive", "neutral", "negative"]

In [ ]:
sample_reviews = [
    "This product is excellent and works perfectly.",
    "The product is okay, nothing special.",
    "Very disappointing item."
]

for r in sample_reviews:
    result = classifier(r, labels)
    print("Review:", r)
    print("Prediction:", result["labels"][0])
    print("Scores:", dict(zip(result["labels"], result["scores"])))
    print("-"*60)

Review: This product is excellent and works perfectly.
Prediction: positive
Scores: {'positive': 0.9958714842796326, 'neutral': 0.0028274324722588062, 'negative': 0.001301125972531736}
------------------------------------------------------------
Review: The product is okay, nothing special.
Prediction: neutral
Scores: {'neutral': 0.9695791006088257, 'positive': 0.018806379288434982, 'negative': 0.01161450520157814}
------------------------------------------------------------
Review: Very disappointing item.
Prediction: negative
Scores: {'negative': 0.9982876181602478, 'positive': 0.000954939576331526, 'neutral': 0.000757489528041333}
------------------------------------------------------------


In [ ]:
predictions = []

for text in df["reviews"]:
    result = classifier(text, labels)
    predictions.append(result["labels"][0])

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [ ]:
print("Accuracy:", accuracy_score(df["sentiment_label"], predictions))

print("\nClassification Report:\n")
print(classification_report(df["sentiment_label"], predictions))

Accuracy: 0.499

Classification Report:

              precision    recall  f1-score   support

    negative       0.72      0.51      0.60       334
     neutral       0.50      0.03      0.05       333
    positive       0.43      0.95      0.59       333

    accuracy                           0.50      1000
   macro avg       0.55      0.50      0.41      1000
weighted avg       0.55      0.50      0.41      1000

